In [1]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub

c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_owner='AmanGupta8423', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow')

mlflow.set_experiment("Hyperparameter Tunning The Tfidf + LogisticRegression")

Accessing as AmanGupta8423

Initialized MLflow to track repo "AmanGupta8423/mlops-mini-project"

Repository AmanGupta8423/mlops-mini-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/0c9f5bef2e864a4c9c9a9391b2923fcf', creation_time=1794744938879, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1794744938879, lifecycle_stage='active', name='Hyperparameter Tunning The Tfidf + LogisticRegression', tags={}, trace_location=None, workspace='default'>

In [3]:
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise



<>:29: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
C:\Users\win11\AppData\Local\Temp\ipykernel_19140\1739671536.py:29: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [6]:
df = normalize_text(df)

x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [7]:
tfidf = TfidfVectorizer(max_features = 1000)
X = tfidf.fit_transform(df['content'])
y = df['sentiment'].astype('int64')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
param_grids = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

In [14]:
with mlflow.start_run(run_name = 'best tunned LR') as parent:
    grid_search = GridSearchCV(LogisticRegression(),param_grid=param_grids, cv = 5, scoring = 'f1', n_jobs = -1)
    grid_search.fit(X_train,y_train)

    for params,mean_score,std_score in zip(grid_search.cv_results_['params'],grid_search.cv_results_['mean_test_score'],grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name = f'LR with params: {params}',nested = True):
            model = LogisticRegression(**params)
            model.fit(X_train,y_train)

            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test,y_pred)
            recall = recall_score(y_test,y_pred)
            precision = precision_score(y_test,y_pred)
            f1 = f1_score(y_test,y_pred)

            mlflow.log_params(params)
            mlflow.log_metrics({
                'accuracy':accuracy,
                'recall':recall,
                'precision':precision,
                'f1_score':f1
            })

            print(f"Mean CV Score: {mean_score}, Std CV Score: {std_score}")
            print(f"Accuracy: {accuracy}")
            print(f"Precision: {precision}")
            print(f"Recall: {recall}")
            print(f"F1 Score: {f1}")

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric('best f1_score',best_score)

    print(f"Best Params: {best_params}")
    print(f"Best F1 Score: {best_score}")

    mlflow.log_artifact('experiment3.ipynb')

    mlflow.sklearn.log_model(sk_model = grid_search.best_estimator_,name = 'model')

c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead 

Mean CV Score: 0.6268106263829344, Std CV Score: 0.011282675647452913
Accuracy: 0.7098795180722891
Precision: 0.7920792079207921
Recall: 0.5517241379310345
F1 Score: 0.6504065040650406
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3/runs/eae81b3557444c4a9135062d602b9774
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3


c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Mean CV Score: 0.7716095050714388, Std CV Score: 0.009646918896452094
Accuracy: 0.7734939759036145
Precision: 0.7716849451645065
Recall: 0.7625615763546798
F1 Score: 0.7670961347869177
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3/runs/6db1edcc9c1742689b2a26ccc4abea94
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3


c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Mean CV Score: 0.7790384457619457, Std CV Score: 0.010552521795528503
Accuracy: 0.776867469879518
Precision: 0.7754491017964071
Recall: 0.7655172413793103
F1 Score: 0.7704511650966782
🏃 View run LR with params: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3/runs/7863d273eb00450692165b83a15288dd
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3


... (2 outputs left out)

c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Mean CV Score: 0.7756445314710643, Std CV Score: 0.011627022408329283
Accuracy: 0.7667469879518072
Precision: 0.7575169738118331
Recall: 0.7694581280788177
F1 Score: 0.7634408602150538
🏃 View run LR with params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3/runs/f3170d7f02df4dc6b133b96fc5550ff9
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3


c:\Users\win11\OneDrive\Desktop\MLops\Project\emotion_detection_final\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Mean CV Score: 0.778793113587952, Std CV Score: 0.010203321537914228
Accuracy: 0.7749397590361445
Precision: 0.7655038759689923
Recall: 0.7783251231527094
F1 Score: 0.7718612603810454
🏃 View run LR with params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3/runs/36a0f2f18d9b4da5a201bdf95b085f0e
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3
Best Params: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
Best F1 Score: 0.7838386908216803
🏃 View run best tunned LR at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3/runs/c9fe41c476544ef38b82bb2a8c00eb0f
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/3
